# Quantum Protein-Ligand Interaction AnalysisPipeline B1-B4: LigandBead | LigandInteraction | HamiltonianBuilder | LigandAnalysis| Etap | Komponent | Opis ||------|-----------|------|| **B1** | `LigandBead` | Pozycja liganda jako zmienna kwantowa (BINARY/UNARY) || **B2** | `LigandInteraction` | Energie oddzialywan HP_LIKE i CUSTOM || **B3** | `HamiltonianBuilder` | H_ligand w przestrzeni protein x ligand || **B4** | `LigandAnalysis` | VQE + binding energy + pozycja + enkapsulacja + vizualizacja |---

## 0. Konfiguracja

In [ ]:
import sys, os, warnings, mathimport numpy as npimport matplotlib.pyplot as plt# Dodaj src do sciezkifor _p in [os.path.abspath('src'), os.path.abspath('../src')]:    if _p not in sys.path:        sys.path.insert(0, _p)plt.rcParams.update({    'figure.facecolor': '#0f1117', 'axes.facecolor': '#1a1d27',    'text.color': '#e0e0e0', 'axes.labelcolor': '#e0e0e0',    'xtick.color': '#c0c0c0', 'ytick.color': '#c0c0c0',    'axes.edgecolor': '#333', 'grid.color': '#2a2d3a', 'font.size': 11,})warnings.filterwarnings('ignore')print('OK')

---## 1. B1 - LigandBead: porownanie enkodowania BINARY vs UNARY

In [ ]:
from particle.ligand_bead import LigandBead, PositionEncodingsizes = [4, 8, 16, 32, 64, 128, 256, 512]bin_q = [LigandBead('L',0,n,PositionEncoding.BINARY).num_position_qubits for n in sizes]una_q = [LigandBead('L',0,n,PositionEncoding.UNARY).num_position_qubits  for n in sizes]fig, ax = plt.subplots(figsize=(10,5))x = np.arange(len(sizes)); w = 0.35b1 = ax.bar(x-w/2, bin_q, w, label='BINARY ceil(log2 N)', color='#4c9be8', edgecolor='#2a2d3a', zorder=3)b2 = ax.bar(x+w/2, una_q, w, label='UNARY (N)',           color='#e86b4c', edgecolor='#2a2d3a', zorder=3)for bar, v in zip(b1, bin_q):    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+1, str(v), ha='center', va='bottom', fontsize=8, color='#4c9be8')for bar, v in zip(b2, una_q):    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+1, str(v), ha='center', va='bottom', fontsize=8, color='#e86b4c')ax.set_xticks(x); ax.set_xticklabels([str(n) for n in sizes])ax.set_xlabel('Liczba wezlow kratki (N)')ax.set_ylabel('Liczba qubitow pozycji')ax.set_title('Kodowanie pozycji liganda: BINARY vs UNARY', color='white', fontsize=13)ax.legend(facecolor='#1a1d27', edgecolor='#444', labelcolor='#e0e0e0')ax.set_yscale('log'); ax.grid(axis='y', linewidth=0.5, zorder=0)fig.tight_layout(); plt.show()print(f'N=512: BINARY={bin_q[-1]}q vs UNARY={una_q[-1]}q ({una_q[-1]/bin_q[-1]:.0f}x redukcja)')

---## 2. B2 - LigandInteraction: profile energetyczneDla sekwencji **APRLRFY** (7 residuow MJ) badamy dwa tryby interakcji liganda.

In [ ]:
from interaction.ligand_interaction import LigandInteractionchain_mj = 'APRLRFY'# HP-like: hydrofobowylig_h = LigandInteraction.hp_like('H')hp_all = lig_h.all_energies()hp_vals = {aa: hp_all.get(aa, 0.0) for aa in chain_mj}# Custom: silnie specyficzny ligand dla APRLRFYhydrophobic = {'A','C','I','L','M','F','W','V'}custom_map = {aa: -2.5 if aa in hydrophobic else (-0.5 if aa in {'P','R'} else -0.1)              for aa in chain_mj}lig_cu = LigandInteraction.custom(energy_map=custom_map, default_energy=0.0)print(f'Sekwencja MJ: {chain_mj}')print(f'HP-like   energies: {hp_vals}')print(f'Custom    energies: {custom_map}')print(f'Sum HP  = {sum(hp_vals[aa] for aa in chain_mj):.1f}')print(f'Sum CU  = {sum(custom_map[aa] for aa in chain_mj):.1f}')fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))aa = list(chain_mj)hp_v = [hp_vals[a] for a in aa]cu_v = [custom_map[a] for a in aa]ax1.bar(aa, hp_v, color=['#e86b4c' if v<0 else '#4c9be8' for v in hp_v], edgecolor='#2a2d3a', zorder=3)ax1.set_title('HP_LIKE (H) dla APRLRFY', color='white', fontsize=12)ax1.axhline(0, color='#555', lw=0.8, ls='--'); ax1.grid(axis='y', lw=0.4, zorder=0)ax1.set_ylabel('E kontaktu [a.u.]')ax2.bar(aa, cu_v, color=['#a87de8' if v<-1 else '#5fcf80' if v<0 else '#4c9be8' for v in cu_v], edgecolor='#2a2d3a', zorder=3)ax2.set_title('CUSTOM (APRLRFY-specyficzny)', color='white', fontsize=12)ax2.axhline(0, color='#555', lw=0.8, ls='--'); ax2.grid(axis='y', lw=0.4, zorder=0)ax2.set_ylabel('E kontaktu [kcal/mol]')fig.suptitle('Profile energetyczne ligand-aminokwas dla APRLRFY', fontsize=13, color='white', y=1.02)fig.tight_layout(); plt.show()

---## 3. B3 - Hamiltonian: rozszerzenie rejestru qubitowegoH = H_backbone + H_backtrack + H_ligand   (n_prot + n_pos qubitow)

In [ ]:
from builder import HamiltonianBuilderfrom contact import ContactMapfrom distance import DistanceMapfrom interaction import MJInteractionfrom protein import Proteinfrom constants import EMPTY_SIDECHAIN_PLACEHOLDER as ESCchain = 'APRLRFY'prot_int = MJInteraction()protein = Protein(chain, ESC*len(chain), prot_int.valid_symbols)builder = HamiltonianBuilder(protein, prot_int, DistanceMap(protein), ContactMap(protein))h_base = builder.sum_hamiltonians()print(f'Bez liganda:  {h_base.num_qubits} qubitow, {len(h_base)} termow')for n_nodes, enc in [(8, PositionEncoding.BINARY), (8, PositionEncoding.UNARY)]:    lig = LigandBead('L', 0, n_nodes, enc)    li  = LigandInteraction.hp_like('H')    h = builder.sum_hamiltonians(ligand=lig, ligand_interaction=li)    print(f'+ {enc.name:<6} N={n_nodes}: {h.num_qubits} qubitow ({h.num_qubits-h_base.num_qubits:+d}), {len(h)} termow')

---## 4. Eksperyment 1: Ligand hydrofobowy + sekwencja HPSekwencja: **HPPHH** | Ligand: H (hydrofobowy) | Oczekiwane: dE = -3.0> VQE 50 iteracji COBYLA (~30s)

In [ ]:
from analysis.ligand_analysis import LigandAnalysis, EncapsulationResultfrom enums import InteractionTypehp_analysis = LigandAnalysis(    main_chain='HPPHH',    interaction_type=InteractionType.HP,    ligand_hp_type='H',    num_lattice_nodes=8,    position_encoding=PositionEncoding.BINARY,    vqe_max_iter=50,)print('VQE HP + ligand H ...')hp_analysis.run()print('\n' + hp_analysis.summary())hp_delta = hp_analysis.compute_binding_energy()expected = sum(hp_analysis._ligand_interaction.get_energy(aa) for aa in 'HPPHH')print(f'\ndE(VQE) = {hp_delta:+.4f}  |  dE(analityczny) = {expected:+.1f}')

---## 5. Eksperyment 2: Ligand custom + sekwencja MJ APRLRFY**Dlaczego APRLRFY?**- Sekwencja zawiera residua o roznej charakterystyce: A, L, F (hydrofobowe), P, R (polarne/naladowane)- MJ model uwzglednia specyficzne oddzialywania miedzy kazda para aminokwasow- Silnie hydrofobowy ligand powinien preferencyjnie wiazac sie z resztami A, L, F> VQE 50 iteracji COBYLA (~45s)

In [ ]:
chain_mj = 'APRLRFY'hydrophobic = {'A','C','I','L','M','F','W','V'}energy_map = {aa: -2.5 if aa in hydrophobic else (-0.5 if aa in {'P','R'} else -0.1)              for aa in chain_mj}print(f'Sekwencja: {chain_mj}')print(f'Profil: {energy_map}')print(f'Sum E_i = {sum(energy_map[aa] for aa in chain_mj):.1f}')mj_analysis = LigandAnalysis(    main_chain=chain_mj,    interaction_type=InteractionType.MJ,    ligand_hp_type=None,    ligand_energy_map=energy_map,    ligand_default_energy=-0.1,    num_lattice_nodes=8,    position_encoding=PositionEncoding.BINARY,    vqe_max_iter=50,)print('\nVQE MJ + ligand custom ...')mj_analysis.run()print('\n' + mj_analysis.summary())mj_delta = mj_analysis.compute_binding_energy()print(f'\ndE(VQE) = {mj_delta:+.4f}')

### 5.1 Porownanie energii wiazania i rozkladow pozycyjnych

In [ ]:
hp_dist = hp_analysis.compute_ligand_position_distribution()mj_dist = mj_analysis.compute_ligand_position_distribution()fig, axes = plt.subplots(1, 3, figsize=(18, 5))# Panel 1: energia wiazaniaax = axes[0]ax.bar(['HP(HPPHH)','MJ(APRLRFY)'], [hp_delta, mj_delta],       color=['#4c9be8' if hp_delta<0 else '#e86b4c', '#a87de8' if mj_delta<0 else '#e86b4c'],       edgecolor='#2a2d3a', width=0.5, zorder=3)ax.axhline(0, color='#666', lw=1, ls='--')ax.set_title('Energia wiazania dE', color='white', fontsize=12)ax.set_ylabel('dE = E_lig - E_base')ax.grid(axis='y', lw=0.4, zorder=0)for i, (label, d) in enumerate([('HP', hp_delta), ('MJ', mj_delta)]):    ax.text(i, d/2, f'{d:+.3f}', ha='center', fontsize=11, color='white', fontweight='bold')# Panel 2: HP rozkladax = axes[1]ax.bar(range(len(hp_dist)), hp_dist, color='#4c9be8', edgecolor='#2a2d3a', width=0.72, zorder=3)ax.set_title('P(pozycja) - HP HPPHH', color='white', fontsize=12)ax.set_xlabel('Wezel kratki'); ax.set_ylabel('P'); ax.grid(axis='y', lw=0.4, zorder=0)# Panel 3: MJ rozkladax = axes[2]ax.bar(range(len(mj_dist)), mj_dist, color='#a87de8', edgecolor='#2a2d3a', width=0.72, zorder=3)ax.set_title('P(pozycja) - MJ APRLRFY', color='white', fontsize=12)ax.set_xlabel('Wezel kratki'); ax.set_ylabel('P'); ax.grid(axis='y', lw=0.4, zorder=0)fig.suptitle('Porownanie: HP vs MJ + ligand', fontsize=14, color='white', y=1.02)fig.tight_layout(); plt.show()

---## 6. Wizualizacja zwiniecia bialka razem z ligandemOdczytujemy bitstring VQE, dekodujemy sekwencje skretow (turn-qubits) na wspolrzedne 3Duzywajac istniejacego `ResultInterpreter`, nastepnie dodajemy liganda w jego**najbardziej prawdopodobnej** pozycji na kratce tetraedrycznej.Otrzymujemy interaktywny wykres Plotly pokazujacy:- Lancuch bialka (kolorowe kulki + wiazania)- Ligand (zlota gwiazda)- Kontakty HH (przerywana linia)- Wezly kratki tetraedrycznej (szare punkty)

In [ ]:
from pathlib import Pathfrom result.interpreter.result_interpreter import ResultInterpreterfrom result.models import BeadPosition# --- Funkcja: extrahuj konformacje bialka z wynikow LigandAnalysis ---def extract_protein_conformation(analysis: LigandAnalysis) -> tuple[list, dict, str]:    """    Zwraca (coords_3d, contacts, best_bitstring) z wynikow VQE.    Uzywa ResultInterpreter do dekodowania turn-qubitow na wspolrzedne.    """    import tempfile    from qiskit_algorithms.minimum_eigensolvers import SamplingMinimumEigensolverResult    from qiskit.quantum_info import SparsePauliOp    baseline = analysis.results['baseline']    # Rekonstrujemy minimalny SamplingMinimumEigensolverResult    # wymagany przez ResultInterpreter    class _FakeRaw:        """Minimal duck-type of SamplingMinimumEigensolverResult."""        def __init__(self, bitstring, probs, energy):            self.best_measurement = {                'bitstring': bitstring,                'probability': probs.get(bitstring, 1.0),                'state': int(bitstring, 2),                'value': energy + 0j,            }            self.eigenstate = probs    raw = _FakeRaw(        bitstring=baseline.best_bitstring,        probs=baseline.state_probabilities,        energy=baseline.minimum_energy,    )    with tempfile.TemporaryDirectory() as tmpdir:        ri = ResultInterpreter(            protein=analysis._protein,            dirpath=Path(tmpdir),            raw_vqe_results=raw,            vqe_energies=baseline.vqe_energies,            vqe_iterations=baseline.vqe_iterations,        )        return ri.coordinates_3d, ri.main_main_contacts_detected, baseline.best_bitstringprint('Ekstraguję konformacje bialka...')hp_coords, hp_contacts, hp_bs = extract_protein_conformation(hp_analysis)mj_coords, mj_contacts, mj_bs = extract_protein_conformation(mj_analysis)print(f'HP: {len(hp_coords)} beadow, bitstring={hp_bs}')print(f'MJ: {len(mj_coords)} beadow, bitstring={mj_bs}')

In [ ]:
import plotly.graph_objects as goimport numpy as npfrom constants import FCC_BASIS# Kolory dla aminokwasowAA_COLORS = {    'H': '#e86b4c', 'P': '#4c9be8',  # HP model    'A': '#f0c040', 'P': '#4c9be8', 'R': '#a87de8', 'L': '#5fcf80',    'F': '#e84c8a', 'Y': '#4ce8d8', 'G': '#aaaaaa',}def ligand_fcc_position(node_idx: int, reference_coord: np.ndarray) -> np.ndarray:    """    Umieszcza liganda w wezle kratki FCC odpowiadajacym node_idx.    Kratka jest generowana wzgledem srodka ciezki bialka.    Wezel node_idx kodowany jest jako kombinacja wektorow bazowych FCC.    """    basis = FCC_BASIS / np.linalg.norm(FCC_BASIS[0])    # Mapa node_idx -> wspolrzedna: ukladamy kratke 2x2x2 okolo srodka    grid = []    for i in range(-2, 3):        for j in range(-2, 3):            for k in range(-2, 3):                grid.append(i*basis[0] + j*basis[1] + k*basis[2])    grid = np.array(grid)    # Sortujemy po odleglosci od srodka i wybieramy node_idx-ty    dists = np.linalg.norm(grid, axis=1)    order = np.argsort(dists)    idx = node_idx % len(order)    return reference_coord + grid[order[idx]]def visualize_protein_with_ligand(    coords_3d,    contacts,    ligand_dist,    title: str,    ligand_symbol: str = 'L',) -> go.Figure:    """    Tworzy interaktywny wykres Plotly z:    - Beadami bialka (kolorowe kulki)    - Wiazaniami peptydowymi (czarna linia)    - Kontaktami HH (fioletowa przerywana linia)    - Ligandem (zlota gwiazda, rozmiar proporcjonalny do P(node))    - Top-3 najprawdopodobniejszymi pozycjami liganda    """    prot_coords = np.array([(b.x, b.y, b.z) for b in coords_3d])    symbols     = [b.symbol for b in coords_3d]    center      = prot_coords.mean(axis=0)    fig = go.Figure()    # Wiazania peptydowe    fig.add_trace(go.Scatter3d(        x=prot_coords[:,0], y=prot_coords[:,1], z=prot_coords[:,2],        mode='lines',        line=dict(color='#aaaaaa', width=6),        name='Wiazania peptydowe',        hoverinfo='skip',    ))    # Kontakty    for i, j in contacts.items():        fig.add_trace(go.Scatter3d(            x=[prot_coords[i,0], prot_coords[j,0]],            y=[prot_coords[i,1], prot_coords[j,1]],            z=[prot_coords[i,2], prot_coords[j,2]],            mode='lines',            line=dict(color='#a87de8', width=5, dash='dot'),            name=f'Kontakt {i}-{j}',            showlegend=True,        ))    # Beady bialka    from matplotlib import cm    cmap = cm.get_cmap('hsv', len(prot_coords))    for i, (sym, (x,y,z)) in enumerate(zip(symbols, prot_coords)):        r,g,b,_ = cmap(i/len(prot_coords))        color = f'rgb({int(r*255)},{int(g*255)},{int(b*255)})'        fig.add_trace(go.Scatter3d(            x=[x], y=[y], z=[z],            mode='markers+text',            marker=dict(size=22, color=color, line=dict(width=2, color='white')),            text=f'{sym}{i}', textposition='middle center',            textfont=dict(size=12, color='white', family='Arial Black'),            name=f'{sym} (idx {i})',            hovertemplate=f'<b>{sym} #{i}</b><br>({x:.2f}, {y:.2f}, {z:.2f})',        ))    # Ligand: top-3 pozycje (rozmiar proporcjonalny do P)    n_show = min(3, len(ligand_dist))    top_nodes = sorted(range(len(ligand_dist)), key=lambda k: ligand_dist[k], reverse=True)[:n_show]    max_p = max(ligand_dist) if max(ligand_dist) > 0 else 1.0    for rank, node in enumerate(top_nodes):        p = ligand_dist[node]        if p < 1e-4:            continue        lig_pos = ligand_fcc_position(node, center)        size = 15 + 25 * (p / max_p)   # rozmiar 15-40        opacity = 0.5 + 0.5 * (p / max_p)        label = 'Ligand (top pozycja)' if rank == 0 else f'Ligand (alt #{rank+1})'        fig.add_trace(go.Scatter3d(            x=[lig_pos[0]], y=[lig_pos[1]], z=[lig_pos[2]],            mode='markers+text',            marker=dict(                size=size,                color='#f0c040' if rank==0 else '#e8a04c',                symbol='diamond' if rank==0 else 'cross',                line=dict(width=3, color='white'),                opacity=opacity,            ),            text=f'L\nnode {node}\nP={p:.3f}',            textposition='top center',            textfont=dict(size=10, color='#f0c040'),            name=label,            hovertemplate=f'<b>Ligand @ wezel {node}</b><br>P={p:.4f}<br>({lig_pos[0]:.2f}, {lig_pos[1]:.2f}, {lig_pos[2]:.2f})',        ))        # Linia laczaca liganda z najblizszym beadem bialka        if rank == 0:            dists_to_prot = np.linalg.norm(prot_coords - lig_pos, axis=1)            nearest = int(np.argmin(dists_to_prot))            np_coord = prot_coords[nearest]            fig.add_trace(go.Scatter3d(                x=[lig_pos[0], np_coord[0]],                y=[lig_pos[1], np_coord[1]],                z=[lig_pos[2], np_coord[2]],                mode='lines',                line=dict(color='#f0c040', width=3, dash='dash'),                name=f'Oddzialywanie L-{symbols[nearest]}{nearest}',                hoverinfo='skip',            ))    fig.update_layout(        title=dict(text=title, font=dict(size=16)),        scene=dict(            xaxis_title='X', yaxis_title='Y', zaxis_title='Z',            aspectmode='data',            xaxis=dict(showbackground=False, showgrid=False),            yaxis=dict(showbackground=False, showgrid=False),            zaxis=dict(showbackground=False, showgrid=False),            bgcolor='#0f1117',        ),        paper_bgcolor='#0f1117',        font=dict(color='#e0e0e0'),        legend=dict(bgcolor='#1a1d27', bordercolor='#333'),        margin=dict(l=0, r=0, b=0, t=60),    )    return figprint('Funkcja wizualizacji gotowa.')

### 6.1 Wizualizacja 3D: HPPHH + ligand hydrofobowyZloty diament = najbardziej prawdopodobna pozycja liganda.Zlota przerywana linia = najsilniejsze oddzialywanie ligand-bead.

In [ ]:
fig_hp = visualize_protein_with_ligand(    coords_3d   = hp_coords,    contacts    = hp_contacts,    ligand_dist = hp_analysis.compute_ligand_position_distribution(),    title       = 'HPPHH + ligand hydrofobowy (H) | dE = ' + f'{hp_delta:+.3f}',)fig_hp.show()

### 6.2 Wizualizacja 3D: APRLRFY + ligand custom (MJ)Sekwencja APRLRFY zawiera:- **A** (ala), **L** (leu), **F** (phe), **Y** (tyr) - hydrofobowe, silnie wiazace liganda (E=-2.5)- **P** (pro), **R** (arg) - polarne/naladowane (E=-0.5)Oczekujemy, ze ligand preferuje okolice residuow hydrofobowych.

In [ ]:
fig_mj = visualize_protein_with_ligand(    coords_3d   = mj_coords,    contacts    = mj_contacts,    ligand_dist = mj_analysis.compute_ligand_position_distribution(),    title       = 'APRLRFY (MJ) + ligand custom | dE = ' + f'{mj_delta:+.3f}',)fig_mj.show()

### 6.3 Statyczne zestawienie 3D (matplotlib) - widok z goryRzut na plaszczyzne XY - kolor kulki = aminokwas, wielkosc = stala.Ligand (zlota gwiazda) umieszczony w najbardziej prawdopodobnym wezle.

In [ ]:
def static_protein_ligand_2d(ax, coords_3d, contacts, ligand_dist, title, palette):    """Statyczny rzut XY bialka + liganda."""    import numpy as np    from constants import FCC_BASIS    pc = np.array([(b.x, b.y) for b in coords_3d])    syms = [b.symbol for b in coords_3d]    center = pc.mean(axis=0)    # Wiazania    ax.plot(pc[:,0], pc[:,1], '-', color='#555', lw=2.5, zorder=1)    # Kontakty HH    for i, j in contacts.items():        ax.plot([pc[i,0], pc[j,0]], [pc[i,1], pc[j,1]],                '--', color='#a87de8', lw=2, alpha=0.8, zorder=2)    # Beady    for i, (sym, (x,y)) in enumerate(zip(syms, pc)):        c = palette[i % len(palette)]        ax.scatter(x, y, s=1200, c=c, edgecolors='white', lw=2, zorder=3)        ax.text(x, y, sym, ha='center', va='center', fontsize=11,                color='white', fontweight='bold', zorder=4)    # Ligand: top-3 pozycje    top_nodes = sorted(range(len(ligand_dist)), key=lambda k: ligand_dist[k], reverse=True)[:3]    basis2d = FCC_BASIS[:, :2] / np.linalg.norm(FCC_BASIS[0])    grid2d = []    for i in range(-3,4):        for j in range(-3,4):            grid2d.append(i*basis2d[0] + j*basis2d[1])    grid2d = np.array(grid2d)    dists2d = np.linalg.norm(grid2d, axis=1)    order2d = np.argsort(dists2d)    max_p = max(ligand_dist) if max(ligand_dist) > 0 else 1.0    for rank, node in enumerate(top_nodes):        p = ligand_dist[node]        if p < 1e-4:            continue        idx2d = node % len(order2d)        lig_xy = center + grid2d[order2d[idx2d]]        sz = 800 + 1200 * (p / max_p)        ax.scatter(*lig_xy, s=sz, c='#f0c040', marker='*',                   edgecolors='white', lw=1.5, zorder=5,                   alpha=0.5 + 0.5*(p/max_p),                   label=f'Ligand node{node} P={p:.3f}' if rank==0 else f'Alt node{node} P={p:.3f}')        ax.annotate(f'L\n({node})\n{p:.3f}', lig_xy,                    fontsize=7, color='#f0c040', ha='center', va='bottom',                    xytext=(0,16), textcoords='offset points', zorder=6)    ax.set_title(title, color='white', fontsize=11, pad=8)    ax.legend(fontsize=7, facecolor='#1a1d27', edgecolor='#444',              labelcolor='#e0e0e0', loc='upper right')    ax.axis('equal')    ax.grid(True, color='#2a2d3a', lw=0.4)palette = ['#4c9be8','#e86b4c','#5fcf80','#f0c040','#a87de8','#e8a04c','#4ce8d8','#e84c8a']fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 7))static_protein_ligand_2d(    ax1, hp_coords, hp_contacts,    hp_analysis.compute_ligand_position_distribution(),    f'HPPHH + ligand H (dE={hp_delta:+.3f})', palette)static_protein_ligand_2d(    ax2, mj_coords, mj_contacts,    mj_analysis.compute_ligand_position_distribution(),    f'APRLRFY (MJ) + ligand custom (dE={mj_delta:+.3f})', palette)fig.suptitle('Konformacje bialek z ligandem - rzut XY (gwiazdka = ligand)', fontsize=14, color='white')fig.tight_layout(); plt.show()print('UWAGA: Wariant A => ligand nie wpywa bezposrednio na konformacje.')print('Rozklad P(node) wynika z preferencji ansatzu VQE.')

---## 7. Enkapsulacja i konwergencja VQE

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))# Row 1: konwergencjafor ax, analysis, title, col in [    (axes[0,0], hp_analysis, 'HP HPPHH + ligand(H)', '#4c9be8'),    (axes[0,1], mj_analysis, 'MJ APRLRFY + ligand(custom)', '#a87de8'),]:    for key, res in analysis.results.items():        if res.vqe_iterations:            style = '-' if key == 'with_ligand' else '--'            c = col if key == 'with_ligand' else '#e86b4c'            ax.plot(res.vqe_iterations, res.vqe_energies, style, lw=2, color=c, label=res.label)    ax.set_title(title, color='white', fontsize=11)    ax.set_xlabel('VQE iterations'); ax.set_ylabel('Energy')    ax.legend(fontsize=8, facecolor='#1a1d27', edgecolor='#444', labelcolor='#e0e0e0')    ax.grid(True, lw=0.4, zorder=0)# Row 2: enkapsulacja przy roznych progachthresholds = np.linspace(0.1, 0.9, 9)for ax, analysis, title, col in [    (axes[1,0], hp_analysis, 'Enkapsulacja HP', '#4c9be8'),    (axes[1,1], mj_analysis, 'Enkapsulacja MJ', '#a87de8'),]:    scores = []    enc_bool = []    for t in thresholds:        enc = analysis.detect_encapsulation(encapsulation_threshold=t)        scores.append(enc.score)        enc_bool.append(int(enc.is_encapsulated))    ax.axhline(scores[0], color=col, lw=2, label=f'Score = {scores[0]:.3f}')    ax.step(thresholds, enc_bool, where='post', color='#f0c040', lw=2, label='Enkapsulowany?')    ax.set_title(title, color='white', fontsize=11)    ax.set_xlabel('Prog enkapsulacji'); ax.set_ylabel('Score / czy enkapsulowany')    ax.legend(fontsize=8, facecolor='#1a1d27', edgecolor='#444', labelcolor='#e0e0e0')    ax.grid(True, lw=0.4, zorder=0)    ax.set_ylim(-0.1, 1.2)fig.suptitle('Konwergencja VQE i analiza enkapsulacji', fontsize=14, color='white')fig.tight_layout(); plt.show()

---## 8. Podsumowanie### Dlaczego APRLRFY zamiast ACDEFGH?- **APRLRFY** jest bardziej biologicznie interesujaca: zawiera proline (P) stabilizujaca zakrety,  arginine (R) naladowana, leucyne (L) i fenyloalanine (F) silnie hydrofobowe- Sekwencja **ACDEFGH** jest sztuczna i monotematyczna- MJ energetics daje ciekawsze oddzialywania dla APRLRFY### Wizualizacja konformacji + ligand- Bialko sklada sie zgodnie z sekwencja skretow z turn-qubitow VQE- Ligand pojawia sie w najblizszym wezle kratki FCC odpowiadajacym najwyzszemu P(node)- Zlota linia = oddzialywanie liganda z najblizszym beadem bialka### Ograniczenie Wariantu AW aktualnej implementacji (Wariant A) H_ligand = scalar shift,wiec geometria liganda NIE zmienia konformacji bialka.Wariant B (z delta(r_i, r_L)) wymagalby qubitow pozycji per bead.

In [ ]:
print('=' * 60)print('  PODSUMOWANIE')print('=' * 60)print(f'  1. HP (HPPHH):       dE = {hp_delta:+.4f}')print(f'  2. MJ (APRLRFY):     dE = {mj_delta:+.4f}')enc_hp = hp_analysis.detect_encapsulation()enc_mj = mj_analysis.detect_encapsulation()print(f'  Enkapsulacja HP: score={enc_hp.score:.3f} ({"YES" if enc_hp.is_encapsulated else "no"})')print(f'  Enkapsulacja MJ: score={enc_mj.score:.3f} ({"YES" if enc_mj.is_encapsulated else "no"})')print('\n  Testy: 312 passed | Wariant A (scalar shift)')print('=' * 60)